In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.preprocessing import RobustScaler

# 1. Re-load Raw Dataset & Apply Feature Engineering
data_path = '../data/raw/PaySim_log.csv'
df = pd.read_csv(data_path)

# Filter Fraud Transaction Types
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# Feature Engineering
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

# One-Hot Encoding
df = pd.get_dummies(df, columns=['type'], drop_first=True)
df.drop(['nameOrig', 'nameDest', 'isFlaggedFraud'], axis=1, inplace=True, errors='ignore')

# Robust Scaling
scaler = RobustScaler()
cols_to_scale = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# 2. Features (X) aur Target (y) Split
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# 3. Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Original Train Class Distribution: {Counter(y_train)}")

# 4. Apply SMOTE with 10% target sampling strategy
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled Train Class Distribution: {Counter(y_train_resampled)}")

# 5. Save Processed Datasets
X_train_resampled.to_csv('../data/processed/X_train_smote.csv', index=False)
y_train_resampled.to_csv('../data/processed/y_train_smote.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("\nPaySim Processed Datasets successfully saved in 'data/processed/' folder!")

Original Train Class Distribution: Counter({0: 367401, 1: 914})
Resampled Train Class Distribution: Counter({0: 367401, 1: 36740})

PaySim Processed Datasets successfully saved in 'data/processed/' folder!
